# Training-Free Bayesianization (TFB) of the Binary Safety Classifier

This notebook implements [ADR 0002](../docs/adr/0002-tfb-uncertainty-for-safety-classification.md) for the binary classifier exported by `training_binary_classifier.ipynb`. It Bayesianizes the trained LoRA adapters without retraining, selects the posterior noise scale $\sigma_q$ on an anchor set, and evaluates the resulting Bayesian model average (BMA) and its uncertainty decomposition.

Notation follows the ADR. For each of the 48 adapted layers, $\mathbf{W}_0 \in \mathbb{R}^{m\times n}$ is the frozen weight, $\Delta\mathbf{W} = s\,\mathbf{B}\mathbf{A}$ with $\mathbf{B} \in \mathbb{R}^{m\times r}$, $\mathbf{A} \in \mathbb{R}^{r\times n}$ and $s = \alpha/r$, and $s\mathbf{B} = \mathbf{U}\,\mathrm{diag}(\mathbf{d})\,\mathbf{V}^\top$ is a compact SVD. The posterior sample is

$$\mathbf{W} = \mathbf{W}_0 + \mathbf{U}\big(\mathbf{U}^\top\Delta\mathbf{W} + \mathbf{Z}\big), \qquad Z_{ij} \sim \mathcal{N}(0, \sigma_q^2),$$

whose covariance is $\mathbf{I}_n \otimes \sigma_q^2\,\mathbf{U}\mathbf{U}^\top$. With $N$ posterior samples giving per-sample probabilities $p_n$ and mean $\bar p$, and $\mathcal{H}_b$ the binary entropy:

$$\mathrm{TU} = \mathcal{H}_b(\bar p), \qquad \mathrm{AU} = \tfrac{1}{N}\textstyle\sum_n \mathcal{H}_b(p_n), \qquad \mathrm{EU} = \mathrm{TU} - \mathrm{AU}.$$

All model outputs are stored as per-sample logits of shape `[N, D, K]` ($D$ examples, $K$ outputs; $K = 1$ here), so the posterior, search, and decomposition code applies unchanged to the multi-label models.

The data are the merged splits the classifier was trained on (the human-labeled rows of Nemotron-Safety-Guard-Dataset-v3 plus all of PolyGuardMix, in seventeen languages), rebuilt from the export's dataset spec by `toy_bald.data.safety_datasets`. Evaluation uses the v3 test split, in its twelve languages. Predictive quality, misclassification detection, distribution shift and the aleatoric check use the full test split, with a per-language breakdown. The analyses that need many extra posterior passes (the sensitivity sweeps, train-vs-test EU and the phase-2 comparison) use fixed language-stratified subsets of `SUBSET_PER_LANGUAGE` prompts per language from the test and training splits.

Distribution shift is measured on two non-safety corpora and on a language-stratified subset of the LLM-labeled jailbreak test prompts, a labeled, safety-relevant shift the model never trained on.

The phase-2 section adds the optional posterior of ADR Decision 2, which also perturbs the weight matrices of the feed-forward head, and compares it with the LoRA-only posterior. A final ablation replaces TFB's low-rank noise by full-rank isotropic noise on the same 48 adapted matrices, testing whether noise outside the adapter's subspace detects shifted inputs better.

In [ ]:
from __future__ import annotations

import json
import math
from collections import OrderedDict
from collections.abc import Callable, Iterator, Sequence
from contextlib import AbstractContextManager, contextmanager
from dataclasses import dataclass
from enum import StrEnum
from pathlib import Path
from typing import Any, Literal, Protocol

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset as HFDataset
from datasets import load_dataset
from peft import PeftModel
from peft.tuners.lora import LoraLayer
from scipy.stats import rankdata
from torch import Tensor
from torch.func import functional_call
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer

from toy_bald.data.batching import bucket_for
from toy_bald.data.safety_datasets import (
    DatasetSpec,
    Source,
    SplitName,
    build_safety_splits,
    make_token_counter,
    stratified_indices,
    take_subset,
)

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

# Export written by training_binary_classifier.ipynb (adapter/, head.pt, config.json).
CHECKPOINT_DIR = Path("../outputs/binary_classifier/me5s_lora_binary/best")
RESULTS_DIR = CHECKPOINT_DIR.parent / "uncertainty"

BATCH_SIZE = 128

# Posterior samples used by the sigma search (ADR Decision 3) and by every
# evaluation (ADR Decision 4), and the sample counts swept in the sensitivity study.
# Samples are drawn in antithetic pairs, so every count must be even.
N_SEARCH_SAMPLES = 10
N_EVAL_SAMPLES = 50
N_SWEEP = (2, 4, 10, 20, 50)

# Sigma search (ADR Decision 3): anchor size, relative NLL tolerance, the relative
# accuracy tolerance of the ablation, bracketing start, and search budget.
ANCHOR_SIZE = 500
EPSILON_REL = 0.003
ACCURACY_TOLERANCE_REL = 0.01
SIGMA_INIT = 1e-3
MAX_BRACKET_DOUBLINGS = 20
BISECTION_STEPS = 8

# Multiples of sigma_star evaluated in the sensitivity study.
SIGMA_GRID_FACTORS = (0.25, 0.5, 1.0, 2.0, 4.0)

# Phase-2 head noise (ADR Decision 2) draws from generators seeded at
# SEED + HEAD_SEED_OFFSET + k, far from the LoRA seeds SEED + k, so the head and
# LoRA noise are independent.
HEAD_SEED_OFFSET = 1_000_000

# The full-rank ablation draws from generators seeded at SEED + FULL_RANK_SEED_OFFSET + k.
FULL_RANK_SEED_OFFSET = 2_000_000

# Resamples of the paired bootstrap that compares the TFB and full-rank posteriors.
BOOTSTRAP_RESAMPLES = 1_000

# Singular directions with d_i <= RANK_TOL * d_1 are dropped (ADR Decision 2).
RANK_TOL = 1e-4

ECE_BINS = 15

# Prompts per language in the fixed test and training subsets used by the sweeps,
# the train-vs-test comparison, and phase 2 (2,400 prompts each for the twelve v3
# languages; the training subset is drawn from the v3 training prompts only).
SUBSET_PER_LANGUAGE = 200

# Out-of-distribution corpora: number of texts kept and minimum characters per text.
OOD_SIZE = 1_000
OOD_MIN_CHARS = 20

# Cache of the merged splits written by the training notebooks (see DatasetSpec).
DATA_CACHE_DIR = Path("../outputs/safety_splits")

# Category that marks the non-unsafe "Needs Caution" annotation.
NEEDS_CAUTION = "Needs Caution"

SEED = 0

In [ ]:
# ---------------------------------------------------------------------------
# Definitions copied verbatim from training_binary_classifier.ipynb. They must
# match the training notebook so the checkpoint and data splits rebuild exactly.
# ---------------------------------------------------------------------------


class AegisLabels(StrEnum):
    SAFE = "safe"
    UNSAFE = "unsafe"


class AegisKeys(StrEnum):
    ID = "id"
    SOURCE = "source"
    LANGUAGE = "language"
    TAG = "tag"
    PROMPT = "prompt"
    RESPONSE = "response"
    PROMPT_LABEL = "prompt_label"
    RESPONSE_LABEL = "response_label"
    VIOLATED_CATEGORIES = "violated_categories"
    PROMPT_LABEL_SOURCE = "prompt_label_source"
    NUM_TOKENS = "num_tokens"


def average_pool(last_hidden_states: Tensor, non_padding_mask: Tensor) -> Tensor:
    """
    Mean-pool token hidden states over non-padded positions.

    :param last_hidden_states:  Shape ``(B, L, D)`` — one vector per token position.
    :param non_padding_mask:    Shape ``(B, L)`` — 1 for real tokens, 0 for padding.
    :return:                    Shape ``(B, D)`` — one dense embedding per input sequence.
    """
    # [B, L, D] — zero out padded positions so they do not contribute to the sum.
    b, l, d = last_hidden_states.shape
    last_hidden = last_hidden_states.masked_fill(~non_padding_mask.reshape(b, l, 1).bool(), 0.0)

    # [B, D] — divide the summed vectors by the count of real tokens per sequence.
    return last_hidden.sum(dim=1) / non_padding_mask.sum(dim=1).reshape(b, 1)


def make_collate_fn(
    tokenizer: Any,
    boundaries: Sequence[int],
):
    """
    Return a DataLoader collate function that tokenizes raw prompts.

    Tokenization happens at collation time, and each batch is padded to the smallest
    bucket length that fits its longest prompt, exactly as during training.

    :param tokenizer:   A HuggingFace ``PreTrainedTokenizer``.
    :param boundaries:  Bucket lengths from the export, ending at the longest prompt allowed.
    :return:            A collate callable for :class:`DataLoader`.
    """

    def _collate(
        batch: list[dict[str, Any]],
    ) -> tuple[dict[str, Tensor], Tensor]:
        texts = [row[AegisKeys.PROMPT] for row in batch]

        # "unsafe" is the positive class: 1.0 for unsafe prompts, 0.0 for safe.
        unsafe_star = torch.tensor(
            [float(row[AegisKeys.PROMPT_LABEL] == AegisLabels.UNSAFE) for row in batch],
            dtype=torch.float32,
        )

        # Tokenize the batch of raw prompts in one call, padded to a bucket length as in
        # training. Every record carries its token count, so nothing is truncated.
        padded_length = bucket_for(max(row[AegisKeys.NUM_TOKENS] for row in batch), boundaries)
        batch_dict: dict[str, Tensor] = tokenizer(
            texts,
            max_length=padded_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        return batch_dict, unsafe_star

    return _collate


def filter_by_token_length(
    dataset: HFDataset,
    tokenizer: Any,
    max_length: int,
    size: int,
) -> HFDataset:
    """
    Keep the leading ``size`` records whose prompt fits within ``max_length`` tokens.

    Records are tokenized without truncation, so the length includes the special
    tokens the encoder actually sees, and each length is stored in ``num_tokens`` for
    the bucketed collate function. The whole split is scanned so ``size`` is
    saturated whenever enough short prompts exist; otherwise every qualifying
    record is returned and a warning is printed.

    :param dataset:     Records with a ``prompt`` column.
    :param tokenizer:   A HuggingFace ``PreTrainedTokenizer``.
    :param max_length:  Maximum allowed token length (inclusive).
    :param size:        Number of records to keep.
    :return:            The first ``min(size, num_qualifying)`` qualifying records,
                        in their original order, with a ``num_tokens`` column.
    """

    # Plain strs (not the enums) keep the functions picklable, so datasets can cache them.
    prompt_key = AegisKeys.PROMPT.value
    length_key = AegisKeys.NUM_TOKENS.value

    def _count(batch: dict[str, list[Any]]) -> dict[str, list[int]]:
        input_ids = tokenizer(batch[prompt_key], truncation=False, verbose=False)["input_ids"]
        return {length_key: [len(ids) for ids in input_ids]}

    def _fits(batch: dict[str, list[Any]]) -> list[bool]:
        return [length <= max_length for length in batch[length_key]]

    qualifying = dataset.map(_count, batched=True).filter(_fits, batched=True)
    if len(qualifying) < size:
        print(
            f"Warning: only {len(qualifying)} of {len(dataset)} records fit within "
            f"{max_length} tokens; requested {size}."
        )
    return qualifying.select(range(min(size, len(qualifying))))


def label_counts(dataset: HFDataset) -> tuple[int, int]:
    """
    Count the safe and unsafe prompts in a split.

    :param dataset: A cleaned split (or subset of one).
    :return:        ``(num_safe, num_unsafe)``.
    """
    labels = list(dataset[AegisKeys.PROMPT_LABEL])
    return labels.count(AegisLabels.SAFE), labels.count(AegisLabels.UNSAFE)



class BinaryFFN(nn.Module):
    """
    Feed-forward head that maps a pooled embedding to a safety classification.

    :param num_layers:      Total number of linear layers (must be >= 2).
    :param input_dim:       Dimensionality of the pooled embedding input.
    :param hidden_dim:      Width of the hidden layers.
    :param weight_init:     Initialization strategy for hidden layers.
    """

    # Number of raw scalar outputs.
    NUM_RAW_OUTPUTS: int = 1

    def __init__(
        self,
        num_layers: int,
        input_dim: int,
        hidden_dim: int,
        weight_init: Literal["x_uniform", "x_normal", "zeros"] = "x_uniform",
    ) -> None:
        super().__init__()
        assert num_layers >= 2, "Need at least an input layer and an output layer."
        assert hidden_dim > 0

        # Select the weight initialization function for hidden layers.
        match weight_init:
            case "x_uniform":
                weight_init_fn = nn.init.xavier_uniform_
            case "x_normal":
                weight_init_fn = nn.init.xavier_normal_
            case "zeros":
                weight_init_fn = nn.init.zeros_
            case _:
                raise ValueError(f"Unknown weight_init: {weight_init!r}")

        # Construct the layer stack: Linear -> [GELU -> Linear] x (num_layers - 1).
        layers: list[tuple[str, nn.Module]] = [("00_Linear", nn.Linear(input_dim, hidden_dim))]
        for i in range(1, num_layers):
            if i < num_layers - 1:
                # Hidden layer: user-selected initialization.
                linear = nn.Linear(hidden_dim, hidden_dim)
                weight_init_fn(linear.weight)
            else:
                # Output layer: zero-initialized weights for a neutral start.
                linear = nn.Linear(hidden_dim, self.NUM_RAW_OUTPUTS)
                nn.init.zeros_(linear.weight)

            nn.init.zeros_(linear.bias)
            layers.extend(
                [
                    (f"{i:02d}_GELU", nn.GELU()),
                    (f"{i:02d}_Linear", linear),
                ]
            )

        self.model = nn.Sequential(OrderedDict(layers))

    def forward(self, x: Tensor) -> Tensor:
        """
        Map a pooled embedding to an unsafe-class logit.

        :param x:   Pooled embedding of shape ``(B, input_dim)``.
        :return:    Logits of shape ``(B,)``.
        """
        # [B, input_dim] -> [B, 1]
        raw = self.model(x)
        # [B, 1] -> [B,]
        return raw.squeeze(1)


class SafetyModel(nn.Module):
    """
    End-to-end safety task-head: LoRA-adapted ME5-small encoder followed
    by a mean-pool and a feed-forward binary classification head.

    :param embedding_model:     A PEFT-wrapped HuggingFace encoder.
    :param ffn_model:           The :class:`BinaryFFN` head.

    """

    def __init__(
        self,
        embedding_model: nn.Module,
        ffn_model: BinaryFFN,
    ) -> None:
        super().__init__()
        self._embedding_model = embedding_model
        self._ffn_model = ffn_model


    def forward(
        self,
        batch_dict: dict[str, Tensor],
        return_attentions: bool = False,
    ) -> Tensor | tuple[Tensor, Tensor, Tensor]:
        """
        Encode the tokenized input and predict if the query is safe or not.

        :param batch_dict:          Tokenizer output dict with ``input_ids`` and
                                    ``attention_mask`` keys.
        :param return_attentions:   When True, also return the final encoder
                                    layer's attention tensor and the attention
                                    mask for heatmap visualization.
        :return:                    Unsafe-class logits of shape ``(B,)``; or,
                                    when ``return_attentions`` is set,
                                    ``(logits, last_layer_attn, attention_mask)``
                                    where ``last_layer_attn`` has shape
                                    ``[B, H, L, L]``.
        """
        # Request per-layer attention weights only when a heatmap is needed;
        # the extra tensors are avoided on the hot training path.
        encoder_kwargs = {"output_attentions": True} if return_attentions else {}

        # Run the LoRA-adapted encoder.
        # [B, L, 384] — one hidden vector per token position.
        emb_outputs = self._embedding_model(**batch_dict, **encoder_kwargs)

        # [B, 384] — mean-pool over non-padded tokens.
        emb_x = average_pool(emb_outputs.last_hidden_state, batch_dict["attention_mask"])

        # [B, 384] -> [B,]
        logits = self._ffn_model(emb_x)

        if not return_attentions:
            return logits

        # emb_outputs.attentions is a tuple of [B, H, L, L] tensors, one per
        # encoder layer. The final layer's map is aggregated downstream into
        # per-token salience for the attention heatmap.
        last_layer_attn = emb_outputs.attentions[-1]
        return logits, last_layer_attn, batch_dict["attention_mask"]


In [ ]:
# ---------------------------------------------------------------------------
# Rebuild the trained model and the exact data splits from the export.
# ---------------------------------------------------------------------------

config = json.loads((CHECKPOINT_DIR / "config.json").read_text())
MAX_LEN = config["max_len"]
UNSAFE_THRESHOLD = config["unsafe_threshold"]
print(f"Checkpoint from epoch {config['epoch']}, val metrics: {config['val_metrics']}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(config["model_id"])

# No attention maps are needed here, so the fused SDPA kernel replaces the eager
# implementation used for training-time heatmaps; the weights are identical.
base_model = AutoModel.from_pretrained(config["model_id"], attn_implementation="sdpa")
lora_model = PeftModel.from_pretrained(base_model, CHECKPOINT_DIR / "adapter")

ffn = BinaryFFN(
    num_layers=config["head"]["num_layers"],
    input_dim=config["hidden_dim"],
    hidden_dim=config["head"]["hidden_dim"],
    weight_init=config["head"]["weight_init"],
)
ffn.load_state_dict(torch.load(CHECKPOINT_DIR / "head.pt", map_location="cpu"))

# Evaluation mode throughout: LoRA dropout must not add a second source of noise.
safety_model = SafetyModel(lora_model, ffn).to(device).eval()

# Exports from before the merged corpus used different splits, which this notebook
# can no longer rebuild.
if "dataset" not in config or "buckets" not in config:
    raise ValueError(f"{CHECKPOINT_DIR} predates the merged safety splits; retrain and re-export the model")
DATASET_SPEC = DatasetSpec.from_json(config["dataset"])
LANGUAGES = DATASET_SPEC.languages
collate_fn = make_collate_fn(tokenizer, tuple(config["buckets"]))

# The training notebook cached the merged splits under this spec, so this loads them;
# take_subset with the exported sizes and seed then reproduces its exact records.
splits, _ = build_safety_splits(DATASET_SPEC, make_token_counter(tokenizer), DATA_CACHE_DIR)
train_dataset = take_subset(splits[SplitName.TRAIN], config["subset_size"], config["seed"])
val_dataset = take_subset(splits[SplitName.VALIDATION], config["val_size"], config["seed"])
test_dataset = take_subset(splits[SplitName.TEST], config["test_size"], config["seed"])

def make_loader(dataset: HFDataset) -> DataLoader:
    """
    Build a deterministic (unshuffled) evaluation loader.

    :param dataset: Records with ``prompt`` and ``prompt_label`` columns.
    :return:        A loader yielding ``(batch_dict, unsafe_star)``.
    """
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)


def unsafe_targets(dataset: HFDataset) -> Tensor:
    """
    Ground-truth unsafe indicators in dataset order.

    :param dataset: A cleaned split (or subset of one).
    :return:        Float targets of shape ``[D, 1]`` (1.0 = unsafe).
    """
    labels = list(dataset[AegisKeys.PROMPT_LABEL])
    return torch.tensor([[float(label == AegisLabels.UNSAFE)] for label in labels])


test_loader = make_loader(test_dataset)
y_test = unsafe_targets(test_dataset)

# Fixed language-stratified subsets for the analyses that need many posterior passes
# (sensitivity sweeps, train-vs-test EU, and phase 2).
test_subset_indices = stratified_indices(test_dataset, SUBSET_PER_LANGUAGE, SEED)
# The training subset is drawn from the v3 training prompts only, so the train-vs-test
# EU comparison contrasts seen and unseen prompts of one source and twelve languages
# rather than also contrasting PolyGuardMix with v3.
train_v3_dataset = train_dataset.select(
    [index for index, source in enumerate(train_dataset[AegisKeys.SOURCE]) if source == Source.NEMOTRON_V3]
)
train_subset_indices = stratified_indices(train_v3_dataset, SUBSET_PER_LANGUAGE, SEED)
test_subset = test_dataset.select(test_subset_indices)
test_subset_loader = make_loader(test_subset)
train_subset_loader = make_loader(train_v3_dataset.select(train_subset_indices))
y_test_subset = y_test[test_subset_indices]

# Jailbreak prompts, never trained on, form a labeled distribution shift.
jailbreak_dataset = take_subset(splits[SplitName.JAILBREAK_TEST], config["test_size"], config["seed"])
jailbreak_subset = jailbreak_dataset.select(stratified_indices(jailbreak_dataset, SUBSET_PER_LANGUAGE, SEED))
y_jailbreak = unsafe_targets(jailbreak_subset)
print(f"train {len(train_dataset)} | val {len(val_dataset)} | test {len(test_dataset)}")
print(
    f"subsets: test {len(test_subset_indices)} | train {len(train_subset_indices)} | "
    f"jailbreak {len(jailbreak_subset)} ({SUBSET_PER_LANGUAGE} per language)"
)

## TFB reparameterization

Every adapted layer is rewritten in the orthonormal parameterization of ADR Decision 2: the $\mathbf{B}$-side factor becomes $\mathbf{U}$ (stored as $\mathbf{U}/s$ because PEFT multiplies by $s$) and the $\mathbf{A}$-side factor becomes the mean $\mathbf{A}'' = \mathbf{U}^\top\Delta\mathbf{W}$. Directions with $d_i \le \tau d_1$ are zeroed. The update $\Delta\mathbf{W}$, and therefore every prediction, is unchanged; the checks below assert this before any noise is added.

In [ ]:
def predict_logits(model: nn.Module, loader: DataLoader) -> Tensor:
    """
    Run one deterministic pass of the model with its current weights.

    :param model:   The safety model (in evaluation mode).
    :param loader:  Loader yielding ``(batch_dict, targets)``.
    :return:        Logits of shape ``[D, K]`` on the CPU.
    """
    outputs: list[Tensor] = []
    with torch.no_grad():
        for batch_dict, _ in loader:
            batch_dict = {k: v.to(device) for k, v in batch_dict.items()}
            logits = model(batch_dict)
            outputs.append(logits.reshape(logits.shape[0], -1).float().cpu())
    return torch.cat(outputs)


@dataclass
class TFBLayer:
    """
    One LoRA layer in TFB's orthonormal parameterization.

    :param name:        Module path inside the PEFT model.
    :param module:      The PEFT ``LoraLayer``.
    :param a_mean:      Posterior mean ``A'' = U^T dW`` of the A-side factor, ``[r, n]``.
    :param keep:        Retained singular directions, ``[r]`` (float 0/1 mask).
    :param spectrum:    Normalized singular values ``d / d_1``, ``[r]``.
    :param original_a:  Trained ``lora_A`` weight, ``[r, n]``.
    :param original_b:  Trained ``lora_B`` weight, ``[m, r]``.
    """

    name: str
    module: LoraLayer
    a_mean: Tensor
    keep: Tensor
    spectrum: Tensor
    original_a: Tensor
    original_b: Tensor


def reparameterize_lora(model: nn.Module, rank_tol: float) -> list[TFBLayer]:
    """
    Rewrite every LoRA layer as ``B'' = U / s`` and ``A'' = U^T dW`` in place.

    :param model:       A model containing PEFT ``LoraLayer`` modules.
    :param rank_tol:    Relative singular-value threshold below which directions are dropped.
    :return:            One :class:`TFBLayer` per adapted layer whose adapter is non-zero.
    """
    layers: list[TFBLayer] = []
    with torch.no_grad():
        for name, module in model.named_modules():
            if not isinstance(module, LoraLayer):
                continue
            lora_a = module.lora_A["default"].weight      # [r, n]
            lora_b = module.lora_B["default"].weight      # [m, r]
            scaling = module.scaling["default"]

            # [m, r] @ [r, n] -> [m, n]
            delta_w = scaling * lora_b @ lora_a
            # U [m, r], d [r]; V is not needed in this parameterization.
            u, d, _ = torch.linalg.svd(scaling * lora_b, full_matrices=False)
            if d[0] <= 0:
                print(f"Skipping {name}: adapter update is zero")
                continue

            keep = (d > rank_tol * d[0]).to(lora_a.dtype)
            u = u * keep
            original_a, original_b = lora_a.clone(), lora_b.clone()
            a_mean = u.T @ delta_w                        # [r, n]
            lora_b.copy_(u / scaling)
            lora_a.copy_(a_mean)

            layers.append(
                TFBLayer(name, module, a_mean, keep, (d / d[0]).cpu(), original_a, original_b)
            )
    return layers


def restore_original_lora(layers: list[TFBLayer]) -> None:
    """
    Write the trained ``lora_A`` and ``lora_B`` weights back into every layer.

    :param layers:  Layers returned by :func:`reparameterize_lora`.
    """
    with torch.no_grad():
        for layer in layers:
            layer.module.lora_A["default"].weight.copy_(layer.original_a)
            layer.module.lora_B["default"].weight.copy_(layer.original_b)


# One test batch is enough to confirm the predictions are untouched.
check_loader = make_loader(test_dataset.select(range(min(BATCH_SIZE, len(test_dataset)))))
logits_before = predict_logits(safety_model, check_loader)

tfb_layers = reparameterize_lora(safety_model, RANK_TOL)

max_update_error = 0.0
for layer in tfb_layers:
    scaling = layer.module.scaling["default"]
    delta_w_original = scaling * layer.original_b @ layer.original_a
    delta_w_new = scaling * layer.module.lora_B["default"].weight @ layer.module.lora_A["default"].weight
    max_update_error = max(max_update_error, (delta_w_new - delta_w_original).abs().max().item())

logits_after = predict_logits(safety_model, check_loader)
max_logit_error = (logits_after - logits_before).abs().max().item()
print(f"{len(tfb_layers)} layers reparameterized")
print(f"max |s B'' A'' - dW| = {max_update_error:.2e}, max logit change = {max_logit_error:.2e}")
assert max_update_error < 1e-5 and max_logit_error < 1e-4

r_eff = [int(layer.keep.sum().item()) for layer in tfb_layers]
print(f"effective ranks: min {min(r_eff)}, max {max(r_eff)} (of {tfb_layers[0].keep.numel()})")

# [layers, r] heatmap of the normalized LoRA spectra.
spectra = torch.stack([layer.spectrum for layer in tfb_layers]).numpy()
fig, ax = plt.subplots(figsize=(8, 10))
image = ax.imshow(spectra, aspect="auto", cmap="viridis", vmin=0.0, vmax=1.0)
ax.set_yticks(range(len(tfb_layers)), [layer.name.split("encoder.")[-1] for layer in tfb_layers], fontsize=6)
ax.set_xlabel("singular direction i")
ax.set_title(r"Normalized LoRA spectra $d_i / d_1$ of $s\mathbf{B}$")
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Posterior sampling with antithetic pairs and common random numbers.
# ---------------------------------------------------------------------------


class PosteriorSampler(Protocol):
    """
    A weight posterior that can temporarily install its ``n``-th sample.
    """

    def sample(self, n: int, sigma: float) -> AbstractContextManager[None]: ...


class TFBPosterior:
    """
    Sampler for the low-rank isotropic Gaussian posterior over all TFB layers.

    Samples come in antithetic pairs: samples ``2k`` and ``2k + 1`` use the
    standard-normal matrices ``+Z_k`` and ``-Z_k``, where ``Z_k`` is drawn from a
    generator seeded with ``seed + k``. Each sample is still an exact draw from the
    posterior, but the average over a pair cancels every odd-order term of the
    Taylor expansion in ``sigma``, so the sample mean of ``Z`` is exactly zero and
    the search objective no longer carries a first-order Monte Carlo term.

    Every candidate ``sigma`` rescales the same matrices (common random numbers),
    which keeps the search objective smooth, and seeding per pair makes sample
    ``n`` identical no matter how many samples a given analysis requests.

    :param layers:  Layers returned by :func:`reparameterize_lora`.
    :param seed:    Base seed of the per-sample generators.
    """

    def __init__(self, layers: list[TFBLayer], seed: int) -> None:
        self.layers = layers
        self.seed = seed
        self._standard_normals: list[list[Tensor]] = []

    def standard_normals(self, n: int) -> list[Tensor]:
        """
        Per-layer standard-normal matrices of sample ``n``, each ``[r, n_in]``.

        :param n:   Sample index; ``n // 2`` selects the pair and ``n % 2`` the sign.
        :return:    One matrix per layer, on the layer's device.
        """
        pair, negate = divmod(n, 2)
        while len(self._standard_normals) <= pair:
            generator = torch.Generator().manual_seed(self.seed + len(self._standard_normals))
            self._standard_normals.append(
                [
                    torch.randn(layer.a_mean.shape, generator=generator).to(layer.a_mean)
                    for layer in self.layers
                ]
            )
        normals = self._standard_normals[pair]
        return [-z for z in normals] if negate else normals

    @contextmanager
    def sample(self, n: int, sigma: float) -> Iterator[None]:
        """
        Temporarily set every A-side factor to ``A'' + sigma * Z_n``.

        :param n:       Sample index.
        :param sigma:   Posterior noise scale ``sigma_q``.
        """
        try:
            with torch.no_grad():
                for layer, z in zip(self.layers, self.standard_normals(n), strict=True):
                    noise = sigma * layer.keep[:, None] * z
                    layer.module.lora_A["default"].weight.copy_(layer.a_mean + noise)
            yield
        finally:
            with torch.no_grad():
                for layer in self.layers:
                    layer.module.lora_A["default"].weight.copy_(layer.a_mean)


def collect_logits(
    model: nn.Module,
    posterior: PosteriorSampler,
    loader: DataLoader,
    sigma: float,
    n_samples: int,
) -> Tensor:
    """
    Per-sample logits of the posterior predictive.

    The outer loop is over posterior samples, so the weights change once per
    sample rather than once per batch. ``sigma == 0`` is the deterministic model
    and returns a single sample.

    :param model:       The safety model (in evaluation mode).
    :param posterior:   The posterior sampler (TFB or full-rank).
    :param loader:      Loader yielding ``(batch_dict, targets)``.
    :param sigma:       Posterior noise scale ``sigma_q``.
    :param n_samples:   Number of posterior samples ``N``; must be even so that
                        every antithetic pair is complete.
    :return:            Logits of shape ``[N, D, K]`` on the CPU.
    """
    if sigma == 0.0:
        return predict_logits(model, loader).unsqueeze(0)
    if n_samples % 2:
        raise ValueError(f"n_samples must be even for antithetic pairs, got {n_samples}")

    samples: list[Tensor] = []
    for n in range(n_samples):
        with posterior.sample(n, sigma):
            samples.append(predict_logits(model, loader))
    return torch.stack(samples)


posterior = TFBPosterior(tfb_layers, SEED)

In [ ]:
# ---------------------------------------------------------------------------
# Uncertainty decomposition and evaluation metrics (ADR Decisions 1 and 5).
# Every function accepts logits [N, D, K] and targets [D, K], so it applies to
# any number of Bernoulli outputs.
# ---------------------------------------------------------------------------

LN2 = math.log(2.0)


def binary_entropy(p: Tensor) -> Tensor:
    """
    Entropy of Bernoulli distributions, in nats.

    :param p:   Success probabilities of any shape.
    :return:    ``-p ln p - (1 - p) ln(1 - p)``, same shape as ``p``.
    """
    p = p.clamp(1e-7, 1.0 - 1e-7)
    return -(p * p.log() + (1.0 - p) * (1.0 - p).log())


@dataclass
class Decomposition:
    """
    Monte Carlo estimates of the predictive mean and the uncertainty decomposition.

    :param p_bar:       BMA probability, ``[D, K]``.
    :param total:       Total uncertainty ``H_b(p_bar)``, ``[D, K]``.
    :param aleatoric:   Aleatoric uncertainty ``mean_n H_b(p_n)``, ``[D, K]``.
    :param epistemic:   Epistemic uncertainty (BALD) ``total - aleatoric``, ``[D, K]``.
    :param logit_variance:  Unbiased sample variance of the logits across posterior
                            samples, ``[D, K]``; zero for a single sample.
    """

    p_bar: Tensor
    total: Tensor
    aleatoric: Tensor
    epistemic: Tensor
    logit_variance: Tensor


def decompose(logits: Tensor) -> Decomposition:
    """
    Decompose the predictive entropy of per-sample logits.

    :param logits:  Per-sample logits, ``[N, D, K]``.
    :return:        The :class:`Decomposition` of the BMA predictive.
    """
    probs = torch.sigmoid(logits)
    p_bar = probs.mean(dim=0)
    total = binary_entropy(p_bar)
    aleatoric = binary_entropy(probs).mean(dim=0)
    # Jensen's inequality guarantees total >= aleatoric; clamp float round-off.
    epistemic = (total - aleatoric).clamp_min(0.0)

    # Posterior disagreement on the logit itself, before the sigmoid; for small
    # noise EU ~= 0.5 * p_bar * (1 - p_bar) * Var[f], so this isolates the second factor.
    # [N, D, K] -> [D, K]
    logit_variance = logits.var(dim=0) if logits.shape[0] > 1 else torch.zeros_like(p_bar)
    return Decomposition(p_bar, total, aleatoric, epistemic, logit_variance)


def gibbs_nll(logits: Tensor, targets: Tensor, mask: Tensor | None = None) -> float:
    """
    Expected negative log-likelihood under the posterior (the sigma-search objective).

    :param logits:  Per-sample logits, ``[N, D, K]``.
    :param targets: Binary targets, ``[D, K]``.
    :param mask:    Optional ``[D, K]`` 0/1 mask of the outputs defined for each
                    example (the hook for masked multi-label heads).
    :return:        ``-E_q[ln p(y | x, theta)]``, summed over outputs and averaged
                    over samples and examples.
    """
    nll = F.binary_cross_entropy_with_logits(
        logits, targets.expand_as(logits), reduction="none"
    )
    if mask is not None:
        nll = nll * mask
    return nll.sum(dim=-1).mean().item()


def bma_nll(logits: Tensor, targets: Tensor) -> float:
    """
    Negative log-likelihood of the Bayesian model average.

    Computed as ``ln p_bar(y) = logsumexp_n ln p_n(y) - ln N`` for numerical stability.

    :param logits:  Per-sample logits, ``[N, D, K]``.
    :param targets: Binary targets, ``[D, K]``.
    :return:        ``-mean_D sum_K ln p_bar(y)``.
    """
    log_lik = -F.binary_cross_entropy_with_logits(
        logits, targets.expand_as(logits), reduction="none"
    )
    log_p_bar = torch.logsumexp(log_lik, dim=0) - math.log(logits.shape[0])
    return -log_p_bar.sum(dim=-1).mean().item()


def reliability_bins(
    p: Tensor, targets: Tensor, num_bins: int
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Bin binary predictions by confidence ``max(p, 1 - p)``.

    :param p:           Predicted probabilities, ``[D]``.
    :param targets:     Binary targets, ``[D]``.
    :param num_bins:    Number of equal-width bins over ``[0, 1]``.
    :return:            Per-bin mean confidence, accuracy, and count, each ``[num_bins]``.
    """
    confidence = torch.maximum(p, 1.0 - p).numpy()
    correct = ((p >= 0.5).float() == targets).float().numpy()
    bin_index = np.minimum((confidence * num_bins).astype(int), num_bins - 1)
    counts = np.bincount(bin_index, minlength=num_bins).astype(float)
    conf_sum = np.bincount(bin_index, weights=confidence, minlength=num_bins)
    acc_sum = np.bincount(bin_index, weights=correct, minlength=num_bins)
    safe_counts = np.maximum(counts, 1.0)
    return conf_sum / safe_counts, acc_sum / safe_counts, counts


def expected_calibration_error(p: Tensor, targets: Tensor, num_bins: int = ECE_BINS) -> float:
    """
    Expected calibration error of binary predictions.

    :param p:           Predicted probabilities, ``[D]``.
    :param targets:     Binary targets, ``[D]``.
    :param num_bins:    Number of equal-width confidence bins.
    :return:            ``sum_k |B_k| / D * |acc(B_k) - conf(B_k)|``.
    """
    conf, acc, counts = reliability_bins(p, targets, num_bins)
    return float(np.sum(counts / counts.sum() * np.abs(acc - conf)))


def auroc(scores: np.ndarray, positives: np.ndarray) -> float:
    """
    Area under the ROC curve via the Mann-Whitney U statistic (ties get mid-ranks).

    :param scores:      Scores where larger means "more likely positive", ``[D]``.
    :param positives:   Boolean positive indicators, ``[D]``.
    :return:            Probability that a random positive outscores a random negative.
    """
    num_pos = int(positives.sum())
    num_neg = len(positives) - num_pos
    if num_pos == 0 or num_neg == 0:
        return float("nan")
    ranks = rankdata(scores)
    return float((ranks[positives].sum() - num_pos * (num_pos + 1) / 2) / (num_pos * num_neg))


def auprc(scores: np.ndarray, positives: np.ndarray) -> float:
    """
    Area under the precision-recall curve as average precision.

    :param scores:      Scores where larger means "more likely positive", ``[D]``.
    :param positives:   Boolean positive indicators, ``[D]``.
    :return:            Mean precision at the rank of each positive.
    """
    order = np.argsort(-scores, kind="stable")
    hits = positives[order].astype(float)
    if hits.sum() == 0:
        return float("nan")
    precision_at_k = np.cumsum(hits) / np.arange(1, len(hits) + 1)
    return float((precision_at_k * hits).sum() / hits.sum())


# --- Sanity checks on synthetic inputs ---
check_generator = torch.Generator().manual_seed(SEED)
random_logits = 3.0 * torch.randn(10, 2_000, 1, generator=check_generator)
random_decomposition = decompose(random_logits)
assert (random_decomposition.epistemic >= 0).all()
assert (random_decomposition.epistemic <= random_decomposition.total + 1e-6).all()
assert (random_decomposition.total <= LN2 + 1e-6).all()
assert decompose(random_logits[:1].expand(10, -1, -1)).epistemic.max() < 1e-5
assert decompose(random_logits[:1].expand(10, -1, -1)).logit_variance.max() == 0.0
assert torch.allclose(random_decomposition.logit_variance, random_logits.var(dim=0))

calibrated_p = torch.rand(200_000, generator=check_generator)
calibrated_y = (torch.rand(200_000, generator=check_generator) < calibrated_p).float()
assert expected_calibration_error(calibrated_p, calibrated_y) < 0.01

separable = np.array([False] * 50 + [True] * 50)
assert auroc(np.arange(100.0), separable) == 1.0 and auroc(-np.arange(100.0), separable) == 0.0
assert auprc(np.arange(100.0), separable) == 1.0
print("Decomposition and metric sanity checks passed.")

## Selecting $\sigma_q$

$\sigma_q^\star$ is the largest noise scale whose expected NLL on the anchor set stays within $\epsilon = 0.003\,\mathcal{L}(0)$ of the deterministic model (ADR Decision 3). The anchor set is 500 training records labeled with the deterministic model's own predictions; the likelihood is unweighted binary cross-entropy. The expectation uses `N_SEARCH_SAMPLES` posterior samples in antithetic pairs, so the Monte Carlo estimate of $\mathcal{L}(\sigma_q) - \mathcal{L}(0)$ has no first-order term in $\sigma_q$.

In [ ]:
def search_sigma(
    objective: Callable[[float], float],
    tolerance: float,
) -> tuple[float, list[tuple[float, float]]]:
    """
    Largest sigma with ``|objective(sigma) - objective(0)| <= tolerance``.

    The bracket starts at ``SIGMA_INIT`` and doubles until the constraint is
    violated; bisection then refines the boundary.

    :param objective:   Anchor metric as a function of sigma.
    :param tolerance:   Allowed absolute change from ``objective(0)``.
    :return:            ``(sigma_star, trace)`` with every evaluated ``(sigma, value)``.
    """
    base = objective(0.0)
    trace = [(0.0, base)]

    def satisfies(sigma: float) -> bool:
        value = objective(sigma)
        trace.append((sigma, value))
        return abs(value - base) <= tolerance

    low, high = 0.0, SIGMA_INIT
    for _ in range(MAX_BRACKET_DOUBLINGS):
        if not satisfies(high):
            break
        low, high = high, 2.0 * high
    else:
        print(
            f"Warning: no sigma up to {low:.3g} violated the tolerance; the anchor metric is "
            "insensitive to this noise (collapsed head?). Returning the largest tested sigma."
        )
        return low, trace

    for _ in range(BISECTION_STEPS):
        middle = 0.5 * (low + high)
        if satisfies(middle):
            low = middle
        else:
            high = middle

    if low == 0.0:
        print("Warning: even SIGMA_INIT / 2**BISECTION_STEPS violated the tolerance")
    return low, trace


# Anchor set: a seeded uniform sample of the training subset with pseudo-labels.
anchor_rng = np.random.default_rng(SEED)
anchor_indices = np.sort(
    anchor_rng.choice(len(train_dataset), size=min(ANCHOR_SIZE, len(train_dataset)), replace=False)
)
anchor_loader = make_loader(train_dataset.select(anchor_indices.tolist()))
anchor_logits_deterministic = collect_logits(safety_model, posterior, anchor_loader, 0.0, 1)
pseudo_labels = (torch.sigmoid(anchor_logits_deterministic[0]) >= UNSAFE_THRESHOLD).float()


def anchor_nll(sigma: float) -> float:
    return gibbs_nll(collect_logits(safety_model, posterior, anchor_loader, sigma, N_SEARCH_SAMPLES), pseudo_labels)


nll_zero = anchor_nll(0.0)
epsilon = EPSILON_REL * nll_zero
sigma_star, nll_trace = search_sigma(anchor_nll, epsilon)
print(f"L(0) = {nll_zero:.4f}, epsilon = {epsilon:.2e}, sigma_star = {sigma_star:.3e}")

trace_sigmas = np.array([s for s, _ in nll_trace[1:]])
trace_changes = np.array([abs(v - nll_zero) for _, v in nll_trace[1:]])
order = np.argsort(trace_sigmas)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(trace_sigmas[order], trace_changes[order], marker="o")
ax.axhline(epsilon, color="tab:red", linestyle="--", label=r"tolerance $\epsilon$")
ax.axvline(sigma_star, color="tab:green", linestyle=":", label=r"$\sigma_q^\star$")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$\sigma_q$")
ax.set_ylabel(r"$|\mathcal{L}(\sigma_q) - \mathcal{L}(0)|$")
ax.set_title("Anchor-set NLL change during the search")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Ablation: an accuracy constraint instead of NLL. Against its own pseudo-labels
# the deterministic model has accuracy 1, so the tolerance is absolute.


def anchor_accuracy(sigma: float) -> float:
    logits = collect_logits(safety_model, posterior, anchor_loader, sigma, N_SEARCH_SAMPLES)
    predictions = (torch.sigmoid(logits) >= UNSAFE_THRESHOLD).float()
    return (predictions == pseudo_labels).float().mean().item()


sigma_star_accuracy, accuracy_trace = search_sigma(
    anchor_accuracy, ACCURACY_TOLERANCE_REL * anchor_accuracy(0.0)
)
print(f"sigma_star (NLL criterion) = {sigma_star:.3e}")
print(f"sigma_star (accuracy criterion) = {sigma_star_accuracy:.3e}")

## Evaluation

Besides TU, AU and EU, every analysis reports the **logit variance** $\mathrm{Var}_\theta[f(x)]$, the sample variance of the pre-sigmoid logit across posterior samples. For small noise, BALD factors as

$$\mathrm{EU}(x) \approx \tfrac{1}{2}\,\bar p(1-\bar p)\,\mathrm{Var}_\theta[f(x)],$$

so EU is the product of closeness to the decision boundary and posterior disagreement. EU is the acquisition score, because it measures the information a label would provide. However, its boundary factor suppresses inputs on which all samples are confident but disagree on *how* confident, which is exactly what off-distribution text tends to look like. The logit variance isolates the disagreement factor and is reported as a diagnostic alongside EU.

In [ ]:
# ---------------------------------------------------------------------------
# Predictive quality on the test set: deterministic vs TFB.
# ---------------------------------------------------------------------------


def predictive_metrics(logits: Tensor, targets: Tensor) -> dict[str, float]:
    """
    Accuracy, BMA NLL, and ECE of a binary BMA predictive.

    :param logits:  Per-sample logits, ``[N, D, 1]``.
    :param targets: Binary targets, ``[D, 1]``.
    :return:        Metric name to value.
    """
    p_bar = decompose(logits).p_bar[:, 0]
    y = targets[:, 0]
    return {
        "accuracy": ((p_bar >= UNSAFE_THRESHOLD).float() == y).float().mean().item(),
        "nll": bma_nll(logits, targets),
        "ece": expected_calibration_error(p_bar, y),
    }


test_logits = {
    "deterministic": collect_logits(safety_model, posterior, test_loader, 0.0, 1),
    "tfb": collect_logits(safety_model, posterior, test_loader, sigma_star, N_EVAL_SAMPLES),
}
quality = {name: predictive_metrics(logits, y_test) for name, logits in test_logits.items()}

print(f"{'model':<15}{'accuracy':>10}{'NLL':>10}{'ECE':>10}")
for name, metrics in quality.items():
    print(f"{name:<15}{metrics['accuracy']:>10.4f}{metrics['nll']:>10.4f}{metrics['ece']:>10.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
for ax, (name, logits) in zip(axes, test_logits.items(), strict=True):
    conf, acc, counts = reliability_bins(decompose(logits).p_bar[:, 0], y_test[:, 0], ECE_BINS)
    filled = counts > 0
    ax.bar(conf[filled], acc[filled] - conf[filled], bottom=conf[filled], width=0.5 / ECE_BINS,
           color="tab:red", alpha=0.6, label="accuracy - confidence")
    ax.plot(conf[filled], acc[filled], marker="o", color="tab:blue", label="accuracy")
    ax.plot([0.5, 1.0], [0.5, 1.0], linestyle="--", color="gray", label="perfect calibration")
    ax.set_xlim(0.5, 1.0)
    ax.set_ylim(0.0, 1.05)
    ax.set_xlabel("confidence max(p, 1 - p)")
    ax.set_title(f"{name}: ECE = {quality[name]['ece']:.4f}")
axes[0].set_ylabel("accuracy")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Misclassification detection: do the uncertainty scores flag wrong predictions?
# ---------------------------------------------------------------------------

test_decomposition = decompose(test_logits["tfb"])
test_errors = (
    ((test_decomposition.p_bar[:, 0] >= UNSAFE_THRESHOLD).float() != y_test[:, 0]).numpy()
)
error_scores = {
    "TU (TFB)": test_decomposition.total[:, 0].numpy(),
    "AU (TFB)": test_decomposition.aleatoric[:, 0].numpy(),
    "EU (TFB)": test_decomposition.epistemic[:, 0].numpy(),
    "Var[f] (TFB)": test_decomposition.logit_variance[:, 0].numpy(),
    "entropy (deterministic)": decompose(test_logits["deterministic"]).total[:, 0].numpy(),
}
misclassification = {
    name: {"auroc": auroc(scores, test_errors), "auprc": auprc(scores, test_errors)}
    for name, scores in error_scores.items()
}

print(f"Test errors: {int(test_errors.sum())} / {len(test_errors)} (AUPRC baseline = error rate {test_errors.mean():.3f})")
print(f"{'score':<26}{'AUROC':>8}{'AUPRC':>8}")
for name, values in misclassification.items():
    print(f"{name:<26}{values['auroc']:>8.4f}{values['auprc']:>8.4f}")

eu = error_scores["EU (TFB)"]
bins = np.linspace(0.0, max(eu.max(), 1e-8), 40)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(eu[~test_errors], bins=bins, alpha=0.6, density=True, label="correct")
ax.hist(eu[test_errors], bins=bins, alpha=0.6, density=True, label="incorrect")
ax.set_xlabel("epistemic uncertainty (nats)")
ax.set_ylabel("density")
ax.set_title("Epistemic uncertainty of correct vs incorrect test predictions")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Per-language breakdown of predictive quality and misclassification detection.
# ---------------------------------------------------------------------------

test_languages = np.asarray(test_dataset[AegisKeys.LANGUAGE])
deterministic_entropy = error_scores["entropy (deterministic)"]
per_language: dict[str, dict[str, float]] = {}

# One row per language, then the pooled row; AUROCs score how well each language's
# errors are ranked by EU and by the deterministic entropy baseline.
print(f"{'language':<10}{'n':>7}{'accuracy':>10}{'NLL':>9}{'ECE':>9}{'mean EU':>11}{'AUROC EU':>10}{'AUROC H':>9}")
for language in [*sorted(set(test_languages)), "all"]:
    mask = np.ones(len(test_languages), dtype=bool) if language == "all" else test_languages == language
    index = torch.from_numpy(np.flatnonzero(mask))
    metrics = predictive_metrics(test_logits["tfb"][:, index], y_test[index])
    metrics.update(
        {
            "n": int(mask.sum()),
            "mean_eu": float(eu[mask].mean()),
            "auroc_eu": auroc(eu[mask], test_errors[mask]),
            "auroc_entropy": auroc(deterministic_entropy[mask], test_errors[mask]),
        }
    )
    per_language[language] = metrics
    print(
        f"{language:<10}{metrics['n']:>7}{metrics['accuracy']:>10.4f}{metrics['nll']:>9.4f}{metrics['ece']:>9.4f}"
        f"{metrics['mean_eu']:>11.2e}{metrics['auroc_eu']:>10.4f}{metrics['auroc_entropy']:>9.4f}"
    )

# Accuracy and EU per language side by side: a language with lower accuracy should
# also carry more epistemic uncertainty if the posterior reflects its weaker fit.
plot_languages = sorted(set(test_languages))
fig, (ax_accuracy, ax_eu) = plt.subplots(1, 2, figsize=(14, 4))
ax_accuracy.bar(plot_languages, [per_language[language]["accuracy"] for language in plot_languages])
ax_accuracy.set_ylim(min(per_language[language]["accuracy"] for language in plot_languages) - 0.05, 1.0)
ax_accuracy.set_title("TFB test accuracy by language")
ax_eu.bar(plot_languages, [per_language[language]["mean_eu"] for language in plot_languages], color="tab:orange")
ax_eu.set_title("Mean epistemic uncertainty by language")
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Distribution shift: epistemic uncertainty on non-safety text and on jailbreak prompts.
# ---------------------------------------------------------------------------


def text_dataset(texts: list[str], size: int) -> HFDataset:
    """
    Wrap raw texts as records the collate function accepts.

    :param texts:   Raw texts.
    :param size:    Number of texts to keep after the token-length filter.
    :return:        Records with a ``prompt`` column and a placeholder ``prompt_label``
                    (these corpora carry no safety label; it is never evaluated).
    """
    records = HFDataset.from_dict(
        {AegisKeys.PROMPT.value: texts, AegisKeys.PROMPT_LABEL.value: [AegisLabels.SAFE.value] * len(texts)}
    )
    return filter_by_token_length(records, tokenizer, MAX_LEN, size)


wikitext = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
wikitext_texts = [
    line.strip()
    for line in wikitext["text"]
    if len(line.strip()) >= OOD_MIN_CHARS and not line.strip().startswith("=")
]
no_robots = load_dataset("HuggingFaceH4/no_robots", split="test")

ood_datasets = {
    "wikitext-2": text_dataset(wikitext_texts, OOD_SIZE),
    "no_robots": text_dataset(list(no_robots["prompt"]), OOD_SIZE),
    # Adversarially framed safety prompts in every language: a labeled shift.
    "jailbreak": jailbreak_subset,
}
ood_logits = {
    name: collect_logits(safety_model, posterior, make_loader(ds), sigma_star, N_EVAL_SAMPLES)
    for name, ds in ood_datasets.items()
}
ood_decompositions = {name: decompose(logits) for name, logits in ood_logits.items()}

# AUROC treats the out-of-distribution texts as positives against the test split.
eu_in = test_decomposition.epistemic[:, 0].numpy()
var_in = test_decomposition.logit_variance[:, 0].numpy()
ood_auroc: dict[str, float] = {}
ood_auroc_logit_variance: dict[str, float] = {}
print(f"{'set':<14}{'size':>6}{'mean EU':>10}{'AUROC EU':>10}{'mean Var[f]':>13}{'AUROC Var[f]':>14}")
print(f"{'test':<14}{len(eu_in):>6}{eu_in.mean():>10.5f}{'-':>10}{var_in.mean():>13.3e}{'-':>14}")
for name, decomposition in ood_decompositions.items():
    eu_out = decomposition.epistemic[:, 0].numpy()
    var_out = decomposition.logit_variance[:, 0].numpy()
    is_ood = np.concatenate([np.zeros(len(eu_in), bool), np.ones(len(eu_out), bool)])
    ood_auroc[name] = auroc(np.concatenate([eu_in, eu_out]), is_ood)
    ood_auroc_logit_variance[name] = auroc(np.concatenate([var_in, var_out]), is_ood)
    print(
        f"{name:<14}{len(eu_out):>6}{eu_out.mean():>10.5f}{ood_auroc[name]:>10.4f}"
        f"{var_out.mean():>13.3e}{ood_auroc_logit_variance[name]:>14.4f}"
    )

# Jailbreak prompts carry labels, so the shift also shows as lost accuracy; the
# uncertainty scores should flag the jailbreak prompts the model gets wrong.
jailbreak_decomposition = ood_decompositions["jailbreak"]
jailbreak_errors = (
    ((jailbreak_decomposition.p_bar[:, 0] >= UNSAFE_THRESHOLD).float() != y_jailbreak[:, 0]).numpy()
)
jailbreak_quality = {
    "accuracy": float(1.0 - jailbreak_errors.mean()),
    "auroc_eu_errors": auroc(jailbreak_decomposition.epistemic[:, 0].numpy(), jailbreak_errors),
    "auroc_tu_errors": auroc(jailbreak_decomposition.total[:, 0].numpy(), jailbreak_errors),
}
print(
    f"\nJailbreak subset: accuracy {jailbreak_quality['accuracy']:.4f}, error-detection AUROC "
    f"EU {jailbreak_quality['auroc_eu_errors']:.4f}, TU {jailbreak_quality['auroc_tu_errors']:.4f}"
)

all_eu = np.concatenate([eu_in] + [d.epistemic[:, 0].numpy() for d in ood_decompositions.values()])
bins = np.linspace(0.0, max(all_eu.max(), 1e-8), 50)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(eu_in, bins=bins, alpha=0.5, density=True, label="test")
for name, decomposition in ood_decompositions.items():
    ax.hist(decomposition.epistemic[:, 0].numpy(), bins=bins, alpha=0.5, density=True, label=name)
ax.set_xlabel("epistemic uncertainty (nats)")
ax.set_ylabel("density")
ax.set_title("Epistemic uncertainty in and out of distribution")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Aleatoric sanity check: prompts with annotator disagreement.
# ---------------------------------------------------------------------------


def has_annotator_disagreement(row: dict[str, Any]) -> bool:
    """
    Flag prompts whose aggregated annotations disagree.

    A record disagrees when it carries ``Needs Caution`` (a non-unsafe vote), or
    when it is labeled safe yet lists an unsafe category (a minority unsafe vote).

    :param row: A cleaned record.
    :return:    True for disagreement prompts.
    """
    raw = row[AegisKeys.VIOLATED_CATEGORIES] or ""
    categories = [category.strip() for category in raw.split(",") if category.strip()]
    if NEEDS_CAUTION in categories:
        return True
    is_safe = row[AegisKeys.PROMPT_LABEL] == AegisLabels.SAFE
    return is_safe and any(category != NEEDS_CAUTION for category in categories)


disagreement = np.array([has_annotator_disagreement(row) for row in test_dataset])
au_test = test_decomposition.aleatoric[:, 0].numpy()
aleatoric_check = {
    "num_disagreement": int(disagreement.sum()),
    "mean_au_disagreement": float(au_test[disagreement].mean()),
    "mean_au_rest": float(au_test[~disagreement].mean()),
    "mean_eu_disagreement": float(eu_in[disagreement].mean()),
    "mean_eu_rest": float(eu_in[~disagreement].mean()),
    "auroc_au": auroc(au_test, disagreement),
    "auroc_eu": auroc(eu_in, disagreement),
}
print(json.dumps(aleatoric_check, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (label, values) in zip(axes, [("aleatoric", au_test), ("epistemic", eu_in)], strict=True):
    bins = np.linspace(0.0, max(values.max(), 1e-8), 40)
    ax.hist(values[~disagreement], bins=bins, alpha=0.6, density=True, label="agreement")
    ax.hist(values[disagreement], bins=bins, alpha=0.6, density=True, label="disagreement")
    ax.set_xlabel(f"{label} uncertainty (nats)")
    ax.set_ylabel("density")
    ax.legend()
fig.suptitle("Test prompts with vs without annotator disagreement")
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Sensitivity to the noise scale and to the number of posterior samples, on the
# language-stratified test subset.
# ---------------------------------------------------------------------------


def summary_metrics(logits: Tensor, targets: Tensor) -> dict[str, float]:
    """
    Predictive metrics plus the mean epistemic uncertainty.

    :param logits:  Per-sample logits, ``[N, D, 1]``.
    :param targets: Binary targets, ``[D, 1]``.
    :return:        Metric name to value.
    """
    return {**predictive_metrics(logits, targets), "mean_eu": decompose(logits).epistemic.mean().item()}


sigma_grid = [factor * sigma_star for factor in SIGMA_GRID_FACTORS]
sigma_sweep = [
    summary_metrics(collect_logits(safety_model, posterior, test_subset_loader, sigma, N_EVAL_SAMPLES), y_test_subset)
    for sigma in sigma_grid
]

# Samples are seeded individually, so the first N of a larger run equal an N-sample run.
logits_max_n = collect_logits(safety_model, posterior, test_subset_loader, sigma_star, max(N_SWEEP))
n_sweep = [summary_metrics(logits_max_n[:n], y_test_subset) for n in N_SWEEP]

metric_names = ["accuracy", "nll", "ece", "mean_eu"]
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for column, metric in enumerate(metric_names):
    axes[0, column].plot(sigma_grid, [m[metric] for m in sigma_sweep], marker="o")
    axes[0, column].axvline(sigma_star, color="tab:green", linestyle=":")
    axes[0, column].set_xscale("log")
    axes[0, column].set_xlabel(r"$\sigma_q$")
    axes[0, column].set_title(metric)
    axes[1, column].plot(N_SWEEP, [m[metric] for m in n_sweep], marker="o", color="tab:orange")
    axes[1, column].set_xscale("log")
    axes[1, column].set_xlabel("posterior samples N")
fig.suptitle(r"Test-subset metrics vs $\sigma_q$ (top, N = %d) and vs N (bottom, $\sigma_q^\star$)" % N_EVAL_SAMPLES)
plt.tight_layout()
plt.show()

## Train vs test epistemic uncertainty

Computed on the language-stratified training and test subsets. Reported for completeness only. Both splits are in-distribution and $\epsilon$ deliberately keeps the posterior tight around the trained weights, so a small or absent gap is expected and does not indicate a problem (ADR Decision 5). The informative tests are the misclassification, distribution-shift, and aleatoric analyses above.

In [ ]:
train_decomposition = decompose(
    collect_logits(safety_model, posterior, train_subset_loader, sigma_star, N_EVAL_SAMPLES)
)
eu_train = train_decomposition.epistemic[:, 0].numpy()

# The test subset's EU is read from the full-test decomposition, which used the same
# posterior samples.
eu_test_subset = eu_in[test_subset_indices]
print(f"mean EU: train subset {eu_train.mean():.5f} | test subset {eu_test_subset.mean():.5f}")

bins = np.linspace(0.0, max(eu_train.max(), eu_test_subset.max(), 1e-8), 40)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(eu_train, bins=bins, alpha=0.6, density=True, label="train subset")
ax.hist(eu_test_subset, bins=bins, alpha=0.6, density=True, label="test subset")
ax.set_xlabel("epistemic uncertainty (nats)")
ax.set_ylabel("density")
ax.legend()
plt.tight_layout()
plt.show()

## Phase 2: Bayesianizing the feed-forward head

The LoRA posterior perturbs only the encoder. The optional phase-2 variant of ADR Decision 2 also perturbs every weight matrix of the head,

$$\mathbf{W}^{\text{head}}_k + \sigma_h\,\rho_k\,\mathbf{\Xi}_k, \qquad k = 1, \dots, L,$$

where $\mathbf{\Xi}_k$ has independent standard-normal entries, $\rho_k$ is the root-mean-square of the entries of $\mathbf{W}^{\text{head}}_k$, and the biases stay deterministic. For an $L$-layer head of width $h$ the matrices have shapes $h\times 384$, $L - 2$ times $h\times h$, and $1\times h$; the current export uses $L = 2$ and $h = 32$. The factor $\rho_k$ makes $\sigma_h$ a *relative* noise level, because the head's layers have different weight scales.

$\sigma_h$ is searched after $\sigma_q$ is fixed. It is the largest value with $|\mathcal{L}(\sigma_q^\star, \sigma_h) - \mathcal{L}(\sigma_q^\star, 0)| \le \epsilon$, using the same anchor set, pseudo-labels and $\epsilon$ as the LoRA search. Head sample $n$ carries the sign of LoRA sample $n$, so the pair $(\mathbf{Z}, \mathbf{\Xi})$ is jointly antithetic. The head runs on cached mean-pooled encoder outputs $\mathbf{h}(x) \in \mathbb{R}^{384}$, so head samples are almost free once the encoder outputs exist.

Three posteriors are compared on the language-stratified test subset under the protocol of Decision 5: LoRA only (phase 1), head only ($\sigma_q = 0$, $\sigma_h^\star$), and LoRA + head ($\sigma_q^\star$, $\sigma_h^\star$).

In [ ]:
# ---------------------------------------------------------------------------
# Phase 2: head posterior and cached encoder outputs (ADR Decision 2).
# ---------------------------------------------------------------------------


def predict_embeddings(model: SafetyModel, loader: DataLoader) -> Tensor:
    """
    Mean-pooled encoder outputs under the model's current adapter weights.

    :param model:   The safety model (in evaluation mode).
    :param loader:  Loader yielding ``(batch_dict, targets)``.
    :return:        Pooled embeddings of shape ``[D, H]`` on the CPU.
    """
    outputs: list[Tensor] = []
    with torch.no_grad():
        for batch_dict, _ in loader:
            batch_dict = {k: v.to(device) for k, v in batch_dict.items()}
            # [B, L, H] -> [B, H], exactly as in SafetyModel.forward.
            hidden = model._embedding_model(**batch_dict).last_hidden_state
            outputs.append(average_pool(hidden, batch_dict["attention_mask"]).float().cpu())
    return torch.cat(outputs)


def collect_embeddings(
    model: SafetyModel,
    posterior: TFBPosterior,
    loader: DataLoader,
    sigma: float,
    n_samples: int,
) -> Tensor:
    """
    Per-sample encoder outputs under the LoRA posterior.

    :param model:       The safety model (in evaluation mode).
    :param posterior:   The TFB posterior sampler.
    :param loader:      Loader yielding ``(batch_dict, targets)``.
    :param sigma:       LoRA noise scale ``sigma_q``; ``0`` returns a single sample.
    :param n_samples:   Number of posterior samples ``N`` (even).
    :return:            Pooled embeddings of shape ``[N, D, H]`` on the CPU.
    """
    if sigma == 0.0:
        return predict_embeddings(model, loader).unsqueeze(0)
    if n_samples % 2:
        raise ValueError(f"n_samples must be even for antithetic pairs, got {n_samples}")

    samples: list[Tensor] = []
    for n in range(n_samples):
        with posterior.sample(n, sigma):
            samples.append(predict_embeddings(model, loader))
    return torch.stack(samples)


class HeadPosterior:
    """
    Sampler for the full-rank isotropic Gaussian over the head's weight matrices.

    Sample ``n`` replaces every weight matrix ``W_k`` by ``W_k + sigma_h * rho_k * Xi_{k,n}``,
    with ``rho_k`` the root-mean-square entry of ``W_k``. As in :class:`TFBPosterior`,
    samples ``2k`` and ``2k + 1`` use ``+Xi_k`` and ``-Xi_k`` drawn from the seed
    ``seed + k``. The trained head is never modified; samples are applied with
    :func:`torch.func.functional_call`.

    :param head:    The trained feed-forward head.
    :param seed:    Base seed of the per-pair generators.
    """

    def __init__(self, head: nn.Module, seed: int) -> None:
        self.seed = seed
        self.means: dict[str, Tensor] = {
            f"{name}.weight": module.weight.detach().clone()
            for name, module in head.named_modules()
            if isinstance(module, nn.Linear)
        }
        self.rms: dict[str, float] = {
            name: weight.pow(2).mean().sqrt().item() for name, weight in self.means.items()
        }
        self._standard_normals: list[dict[str, Tensor]] = []

    def standard_normals(self, n: int) -> dict[str, Tensor]:
        """
        Standard-normal matrices of sample ``n``, keyed like :attr:`means`.

        :param n:   Sample index; ``n // 2`` selects the pair and ``n % 2`` the sign.
        :return:    One matrix per weight, on the weight's device.
        """
        pair, negate = divmod(n, 2)
        while len(self._standard_normals) <= pair:
            generator = torch.Generator().manual_seed(self.seed + len(self._standard_normals))
            self._standard_normals.append(
                {
                    name: torch.randn(mean.shape, generator=generator).to(mean)
                    for name, mean in self.means.items()
                }
            )
        normals = self._standard_normals[pair]
        return {name: -z for name, z in normals.items()} if negate else normals

    def weights(self, n: int, sigma: float) -> dict[str, Tensor]:
        """
        Weight matrices of head sample ``n`` at relative noise level ``sigma``.

        :param n:       Sample index.
        :param sigma:   Relative head noise scale ``sigma_h``.
        :return:        Parameter name to sampled weight, for ``functional_call``.
        """
        if sigma == 0.0:
            return self.means
        normals = self.standard_normals(n)
        return {
            name: mean + sigma * self.rms[name] * normals[name] for name, mean in self.means.items()
        }


def head_logits(
    head: nn.Module,
    posterior: HeadPosterior,
    embeddings: Tensor,
    sigma: float,
    n_samples: int,
) -> Tensor:
    """
    Per-sample head logits on cached encoder outputs.

    A single encoder sample (deterministic encoder) is shared by every head sample.
    Otherwise head sample ``n`` reads encoder sample ``n``, pairing the LoRA and
    head noise of the same index. With ``sigma == 0`` the head is deterministic and
    one logit sample is returned per encoder sample.

    :param head:        The trained feed-forward head.
    :param posterior:   The head posterior sampler.
    :param embeddings:  Encoder outputs, ``[N_e, D, H]`` with ``N_e`` 1 or ``n_samples``.
    :param sigma:       Relative head noise scale ``sigma_h``.
    :param n_samples:   Number of head samples (even when ``sigma > 0``).
    :return:            Logits of shape ``[N, D, K]`` on the CPU.
    """
    num_encoder = embeddings.shape[0]
    if sigma == 0.0:
        n_samples = num_encoder
    elif n_samples % 2:
        raise ValueError(f"n_samples must be even for antithetic pairs, got {n_samples}")
    if num_encoder not in (1, n_samples):
        raise ValueError(f"{num_encoder} encoder samples cannot pair with {n_samples} head samples")

    samples: list[Tensor] = []
    with torch.no_grad():
        for n in range(n_samples):
            # [D, H] -> [D] (BinaryFFN squeezes its single output) -> [D, K]
            x = embeddings[n if num_encoder > 1 else 0].to(device)
            logits = functional_call(head, posterior.weights(n, sigma), (x,))
            samples.append(logits.reshape(x.shape[0], -1).float().cpu())
    return torch.stack(samples)


head_posterior = HeadPosterior(ffn, SEED + HEAD_SEED_OFFSET)
for name, mean in head_posterior.means.items():
    print(f"{name:<20}{str(tuple(mean.shape)):>12}  rms = {head_posterior.rms[name]:.4f}")

# The cached-embedding route with a noise-free head must reproduce the full model,
# and paired head samples must carry exactly opposite noise.
check_embeddings = collect_embeddings(safety_model, posterior, check_loader, 0.0, 1)
cached_route_error = (head_logits(ffn, head_posterior, check_embeddings, 0.0, 1)[0] - logits_before).abs().max().item()
print(f"max logit change through cached embeddings = {cached_route_error:.2e}")
assert cached_route_error < 1e-4
assert all(
    torch.equal(head_posterior.standard_normals(0)[name], -head_posterior.standard_normals(1)[name])
    for name in head_posterior.means
)

In [ ]:
# ---------------------------------------------------------------------------
# Search sigma_h with the LoRA posterior fixed at sigma_q* (ADR Decision 2).
# ---------------------------------------------------------------------------

# Anchor encoder outputs under the LoRA posterior are computed once and reused by
# every candidate sigma_h, which only reruns the head.
anchor_embeddings = collect_embeddings(safety_model, posterior, anchor_loader, sigma_star, N_SEARCH_SAMPLES)


def anchor_nll_head(sigma_h: float) -> float:
    logits = head_logits(ffn, head_posterior, anchor_embeddings, sigma_h, N_SEARCH_SAMPLES)
    return gibbs_nll(logits, pseudo_labels)


nll_lora = anchor_nll_head(0.0)
sigma_h_star, head_trace = search_sigma(anchor_nll_head, epsilon)
print(f"L(0) = {nll_zero:.4f}, L(sigma_q*, 0) = {nll_lora:.4f}, epsilon = {epsilon:.2e}")
print(f"sigma_h_star = {sigma_h_star:.3e} (relative to each layer's weight rms)")

head_sigmas = np.array([s for s, _ in head_trace[1:]])
head_changes = np.array([abs(v - nll_lora) for _, v in head_trace[1:]])
order = np.argsort(head_sigmas)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(head_sigmas[order], np.maximum(head_changes[order], 1e-12), marker="o")
ax.axhline(epsilon, color="tab:red", linestyle="--", label=r"tolerance $\epsilon$")
ax.axvline(sigma_h_star, color="tab:green", linestyle=":", label=r"$\sigma_h^\star$")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$\sigma_h$")
ax.set_ylabel(r"$|\mathcal{L}(\sigma_q^\star, \sigma_h) - \mathcal{L}(\sigma_q^\star, 0)|$")
ax.set_title(r"Anchor-set NLL change during the $\sigma_h$ search")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Compare the LoRA-only, head-only, and LoRA + head posteriors (ADR Decision 5).
# ---------------------------------------------------------------------------

PHASE2_LABELS = {
    "deterministic": "deterministic",
    "lora": "LoRA (phase 1)",
    "head": "head only",
    "lora_head": "LoRA + head",
}
PHASE2_VARIANTS = ["lora", "head", "lora_head"]

# Encoder outputs for the test subset and out-of-distribution sets, deterministic and under
# the LoRA posterior; every head variant is then a cheap pass over these caches.
eval_sets = {"test": test_subset, **ood_datasets}
phase2_logits: dict[str, dict[str, Tensor]] = {}
for set_name, dataset in eval_sets.items():
    loader = make_loader(dataset)
    deterministic_outputs = collect_embeddings(safety_model, posterior, loader, 0.0, 1)
    lora_outputs = collect_embeddings(safety_model, posterior, loader, sigma_star, N_EVAL_SAMPLES)
    phase2_logits[set_name] = {
        "deterministic": head_logits(ffn, head_posterior, deterministic_outputs, 0.0, 1),
        "lora": head_logits(ffn, head_posterior, lora_outputs, 0.0, N_EVAL_SAMPLES),
        "head": head_logits(ffn, head_posterior, deterministic_outputs, sigma_h_star, N_EVAL_SAMPLES),
        "lora_head": head_logits(ffn, head_posterior, lora_outputs, sigma_h_star, N_EVAL_SAMPLES),
    }

# The cached route reuses the LoRA noise of phase 1, so it must reproduce its logits.
phase1_error = (phase2_logits["test"]["lora"] - test_logits["tfb"][:, test_subset_indices]).abs().max().item()
print(f"max |phase-2 LoRA logits - phase-1 TFB logits| = {phase1_error:.2e}")
assert phase1_error < 1e-4

phase2_decompositions = {
    set_name: {variant: decompose(logits) for variant, logits in by_variant.items()}
    for set_name, by_variant in phase2_logits.items()
}
test_by_variant = phase2_decompositions["test"]

# Predictive quality and uncertainty magnitudes on the test subset.
phase2_quality = {
    variant: {
        **predictive_metrics(phase2_logits["test"][variant], y_test_subset),
        "mean_eu": test_by_variant[variant].epistemic.mean().item(),
        "mean_logit_variance": test_by_variant[variant].logit_variance.mean().item(),
    }
    for variant in ["deterministic", *PHASE2_VARIANTS]
}
print(f"\n{'posterior':<16}{'accuracy':>10}{'NLL':>10}{'ECE':>10}{'mean EU':>11}{'mean Var[f]':>13}")
for variant, metrics in phase2_quality.items():
    print(
        f"{PHASE2_LABELS[variant]:<16}{metrics['accuracy']:>10.4f}{metrics['nll']:>10.4f}"
        f"{metrics['ece']:>10.4f}{metrics['mean_eu']:>11.5f}{metrics['mean_logit_variance']:>13.3e}"
    )


def uncertainty_scores(decomposition: Decomposition) -> dict[str, np.ndarray]:
    return {
        "EU": decomposition.epistemic[:, 0].numpy(),
        "Var[f]": decomposition.logit_variance[:, 0].numpy(),
    }


# Misclassification detection, with errors defined by each variant's own BMA prediction.
phase2_misclassification: dict[str, dict[str, dict[str, float]]] = {}
for variant in PHASE2_VARIANTS:
    decomposition = test_by_variant[variant]
    errors = ((decomposition.p_bar[:, 0] >= UNSAFE_THRESHOLD).float() != y_test_subset[:, 0]).numpy()
    phase2_misclassification[variant] = {
        score: {"auroc": auroc(values, errors), "auprc": auprc(values, errors), "error_rate": float(errors.mean())}
        for score, values in uncertainty_scores(decomposition).items()
    }

# Distribution shift: each out-of-distribution set against the test subset.
phase2_ood_auroc: dict[str, dict[str, dict[str, float]]] = {}
for variant in PHASE2_VARIANTS:
    scores_in = uncertainty_scores(test_by_variant[variant])
    phase2_ood_auroc[variant] = {}
    for set_name in ood_datasets:
        scores_out = uncertainty_scores(phase2_decompositions[set_name][variant])
        is_ood = np.concatenate([np.zeros(len(scores_in["EU"]), bool), np.ones(len(scores_out["EU"]), bool)])
        phase2_ood_auroc[variant][set_name] = {
            score: auroc(np.concatenate([scores_in[score], scores_out[score]]), is_ood) for score in scores_in
        }

# Aleatoric sanity check: AU should still track annotator disagreement.
phase2_aleatoric_auroc = {
    variant: auroc(test_by_variant[variant].aleatoric[:, 0].numpy(), disagreement[test_subset_indices])
    for variant in PHASE2_VARIANTS
}

print("\nMisclassification detection on the test subset")
print(f"{'posterior':<16}{'score':<8}{'error rate':>11}{'AUROC':>8}{'AUPRC':>8}")
for variant in PHASE2_VARIANTS:
    for score, metrics in phase2_misclassification[variant].items():
        print(
            f"{PHASE2_LABELS[variant]:<16}{score:<8}{metrics['error_rate']:>11.3f}"
            f"{metrics['auroc']:>8.4f}{metrics['auprc']:>8.4f}"
        )

ood_names = list(ood_datasets)
print("\nOut-of-distribution AUROC vs the test subset")
print(f"{'posterior':<16}{'score':<8}" + "".join(f"{name:>14}" for name in ood_names))
for variant in PHASE2_VARIANTS:
    for score in ("EU", "Var[f]"):
        columns = "".join(f"{phase2_ood_auroc[variant][name][score]:>14.4f}" for name in ood_names)
        print(f"{PHASE2_LABELS[variant]:<16}{score:<8}{columns}")
print("\nAU vs annotator disagreement AUROC: " + ", ".join(
    f"{PHASE2_LABELS[v]} {phase2_aleatoric_auroc[v]:.4f}" for v in PHASE2_VARIANTS
))

# Logit variance in and out of distribution, one panel per posterior, on a log scale
# because the three posteriors differ by orders of magnitude.
fig, axes = plt.subplots(1, len(PHASE2_VARIANTS), figsize=(6 * len(PHASE2_VARIANTS), 4))
for ax, variant in zip(axes, PHASE2_VARIANTS, strict=True):
    log_variances = {
        set_name: np.log10(by_variant[variant].logit_variance[:, 0].numpy() + 1e-12)
        for set_name, by_variant in phase2_decompositions.items()
    }
    pooled = np.concatenate(list(log_variances.values()))
    bins = np.linspace(pooled.min(), max(pooled.max(), pooled.min() + 1e-6), 50)
    for set_name, values in log_variances.items():
        ax.hist(values, bins=bins, alpha=0.5, density=True, label="test subset" if set_name == "test" else set_name)
    ax.set_xlabel(r"$\log_{10}\,\mathrm{Var}_\theta[f(x)]$")
    ax.set_title(PHASE2_LABELS[variant])
axes[0].set_ylabel("density")
axes[0].legend()
fig.suptitle("Logit variance in and out of distribution")
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Persist the results and restore the trained adapters.
# ---------------------------------------------------------------------------

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
summary = {
    "checkpoint_epoch": config["epoch"],
    "n_search_samples": N_SEARCH_SAMPLES,
    "n_eval_samples": N_EVAL_SAMPLES,
    "anchor_size": len(anchor_indices),
    "nll_zero": nll_zero,
    "epsilon": epsilon,
    "sigma_star": sigma_star,
    "sigma_star_accuracy": sigma_star_accuracy,
    "search_trace": nll_trace,
    "effective_ranks": r_eff,
    "test_quality": quality,
    "per_language": per_language,
    "subset_per_language": SUBSET_PER_LANGUAGE,
    "misclassification": misclassification,
    "ood_auroc": ood_auroc,
    "ood_auroc_logit_variance": ood_auroc_logit_variance,
    "jailbreak_quality": jailbreak_quality,
    "aleatoric_check": aleatoric_check,
    "sigma_sweep": dict(zip(map(str, sigma_grid), sigma_sweep, strict=True)),
    "n_sweep": dict(zip(map(str, N_SWEEP), n_sweep, strict=True)),
    "head": {
        "sigma_h_star": sigma_h_star,
        "nll_lora": nll_lora,
        "search_trace": head_trace,
        "weight_rms": head_posterior.rms,
        "quality": phase2_quality,
        "misclassification": phase2_misclassification,
        "ood_auroc": phase2_ood_auroc,
        "aleatoric_auroc": phase2_aleatoric_auroc,
    },
}
(RESULTS_DIR / "summary.json").write_text(json.dumps(summary, indent=2))

score_sets = {"train_subset": train_decomposition, "test": test_decomposition, **ood_decompositions}
np.savez(
    RESULTS_DIR / "scores.npz",
    **{
        f"{name}_{field}": getattr(decomposition, field)[:, 0].numpy()
        for name, decomposition in score_sets.items()
        for field in ("p_bar", "total", "aleatoric", "epistemic", "logit_variance")
    },
    **{
        f"phase2_{variant}_{set_name}_{field}": getattr(decomposition, field)[:, 0].numpy()
        for set_name, by_variant in phase2_decompositions.items()
        for variant, decomposition in by_variant.items()
        for field in ("p_bar", "epistemic", "logit_variance")
    },
)
print(f"Wrote {RESULTS_DIR / 'summary.json'} and {RESULTS_DIR / 'scores.npz'}")

restore_original_lora(tfb_layers)
restored_error = (predict_logits(safety_model, check_loader) - logits_before).abs().max().item()
print(f"Restored trained adapters (max logit change {restored_error:.2e})")

## Ablation: full-rank noise on the adapted matrices

TFB confines the noise of each adapted layer to the $r = 16$ directions $\mathbf{U}$ the adapter learned. The out-of-distribution results suggest why that fails off-distribution: text that barely activates those directions barely feels the noise. This ablation spreads the noise over all directions of the same 48 matrices,

$$\mathbf{W} = \mathbf{W}_0 + \Delta\mathbf{W} + \sigma_f\,\rho_\ell\,\mathbf{\Xi}, \qquad \Xi_{ij} \sim \mathcal{N}(0, 1),$$

with covariance $\mathbf{I}_n \otimes \sigma_f^2\rho_\ell^2\,\mathbf{I}_m$ instead of TFB's $\mathbf{I}_n \otimes \sigma_q^2\,\mathbf{U}\mathbf{U}^\top$. Here $\rho_\ell$ is the root-mean-square entry of layer $\ell$'s merged weight $\mathbf{W}_0 + \Delta\mathbf{W}$, so $\sigma_f$ is a relative noise level, as for the head. $\sigma_f^\star$ comes from the same search, anchor set, pseudo-labels and tolerance $\epsilon$ as $\sigma_q^\star$, so both posteriors spend the same NLL budget and differ only in which inputs absorb it. Samples come in antithetic pairs. Unlike TFB, this family has no variational-inference guarantee; it is a heuristic perturbation posterior.

Both posteriors are compared on the language-stratified test subset and the distribution-shift sets. Differences in AUROC carry 95% paired bootstrap intervals: prompts are resampled with replacement, within each group, and both posteriors are scored on the same resample.

In [ ]:
# ---------------------------------------------------------------------------
# Full-rank isotropic noise on the 48 adapted matrices (ablation of ADR Decision 2).
# ---------------------------------------------------------------------------


class FullRankPosterior:
    """
    Sampler for full-rank isotropic noise on the merged weight of every adapted layer.

    Sample ``n`` adds ``sigma * rho_l * Xi_{l,n}`` to the frozen base weight ``W_0`` of
    layer ``l``, so the layer computes ``W_0 + dW + noise`` while the adapter keeps
    its trained update ``dW``. ``rho_l`` is the root-mean-square entry of
    ``W_0 + dW``. Samples ``2k`` and ``2k + 1`` use ``+Xi_k`` and ``-Xi_k`` drawn from
    the seed ``seed + k``. The noise is regenerated for every sample instead of
    cached, because one sample holds about seven million entries. The base weights
    are restored by copying saved clones, which is exact.

    :param layers:  Layers returned by :func:`reparameterize_lora`.
    :param seed:    Base seed of the per-pair generators.
    """

    def __init__(self, layers: list[TFBLayer], seed: int) -> None:
        self.seed = seed
        self.base_layers: list[nn.Linear] = [layer.module.base_layer for layer in layers]
        self.base_weights: list[Tensor] = [base.weight.detach().clone() for base in self.base_layers]

        # [m, r] @ [r, n] -> [m, n]: each layer's merged weight sets its noise scale.
        self.rms: list[float] = []
        with torch.no_grad():
            for layer, base_weight in zip(layers, self.base_weights, strict=True):
                scaling = layer.module.scaling["default"]
                delta_w = scaling * layer.module.lora_B["default"].weight @ layer.module.lora_A["default"].weight
                self.rms.append((base_weight + delta_w).pow(2).mean().sqrt().item())

    @contextmanager
    def sample(self, n: int, sigma: float) -> Iterator[None]:
        """
        Temporarily set every base weight to ``W_0 + sigma * rho_l * Xi_{l,n}``.

        :param n:       Sample index; ``n // 2`` selects the pair and ``n % 2`` the sign.
        :param sigma:   Relative noise scale ``sigma_f``.
        """
        pair, negate = divmod(n, 2)
        sign = -1.0 if negate else 1.0
        generator = torch.Generator().manual_seed(self.seed + pair)
        try:
            with torch.no_grad():
                for base, base_weight, rms in zip(self.base_layers, self.base_weights, self.rms, strict=True):
                    # [m, n] standard normals, drawn on the CPU so the stream is device-independent.
                    xi = torch.randn(base_weight.shape, generator=generator).to(base_weight)
                    base.weight.copy_(base_weight + (sign * sigma * rms) * xi)
            yield
        finally:
            with torch.no_grad():
                for base, base_weight in zip(self.base_layers, self.base_weights, strict=True):
                    base.weight.copy_(base_weight)


def bootstrap_difference(
    statistic_a: Callable[[list[np.ndarray]], float],
    statistic_b: Callable[[list[np.ndarray]], float],
    group_sizes: Sequence[int],
    num_resamples: int,
    seed: int,
) -> dict[str, float]:
    """
    Paired bootstrap of ``statistic_a - statistic_b``.

    Each resample draws, with replacement and within each group, as many indices as
    the group has; both statistics are evaluated on the same indices, so the
    interval reflects the difference on shared prompts rather than two independent
    uncertainties.

    :param statistic_a:     Statistic of posterior A given one index array per group.
    :param statistic_b:     Statistic of posterior B, with the same signature.
    :param group_sizes:     Number of items in each group.
    :param num_resamples:   Bootstrap resamples.
    :param seed:            Seed of the resampling.
    :return:                Observed difference and the 2.5% and 97.5% quantiles.
    """
    rng = np.random.default_rng(seed)
    full = [np.arange(size) for size in group_sizes]
    differences = []
    for _ in range(num_resamples):
        indices = [rng.integers(0, size, size) for size in group_sizes]
        differences.append(statistic_a(indices) - statistic_b(indices))
    low, high = np.nanquantile(differences, [0.025, 0.975])
    return {"difference": statistic_a(full) - statistic_b(full), "ci_low": float(low), "ci_high": float(high)}


full_rank_posterior = FullRankPosterior(tfb_layers, SEED + FULL_RANK_SEED_OFFSET)

# Same anchor set, pseudo-labels, tolerance and sample count as the sigma_q search.
def anchor_nll_full_rank(sigma: float) -> float:
    logits = collect_logits(safety_model, full_rank_posterior, anchor_loader, sigma, N_SEARCH_SAMPLES)
    return gibbs_nll(logits, pseudo_labels)


sigma_f_star, full_rank_trace = search_sigma(anchor_nll_full_rank, epsilon)
print(f"sigma_f_star = {sigma_f_star:.3e} (relative to each layer's weight rms); sigma_q_star = {sigma_star:.3e}")

# Removing the noise must restore the base weights exactly.
assert all(
    torch.equal(base.weight, saved)
    for base, saved in zip(full_rank_posterior.base_layers, full_rank_posterior.base_weights, strict=True)
)

In [ ]:
# ---------------------------------------------------------------------------
# Compare the full-rank and TFB posteriors on the test subset and the shift sets.
# ---------------------------------------------------------------------------

# TFB's logits on these sets already exist, from the same posterior samples.
ablation_logits: dict[str, dict[str, Tensor]] = {
    "test": {
        "tfb": test_logits["tfb"][:, test_subset_indices],
        "full_rank": collect_logits(safety_model, full_rank_posterior, test_subset_loader, sigma_f_star, N_EVAL_SAMPLES),
    }
}
for set_name, dataset in ood_datasets.items():
    ablation_logits[set_name] = {
        "tfb": ood_logits[set_name],
        "full_rank": collect_logits(safety_model, full_rank_posterior, make_loader(dataset), sigma_f_star, N_EVAL_SAMPLES),
    }
ablation_decompositions = {
    set_name: {variant: decompose(logits) for variant, logits in by_variant.items()}
    for set_name, by_variant in ablation_logits.items()
}
ABLATION_VARIANTS = {"tfb": "TFB (low-rank)", "full_rank": "full-rank"}

# Predictive quality and uncertainty magnitudes on the test subset.
ablation_quality = {
    variant: {
        **predictive_metrics(ablation_logits["test"][variant], y_test_subset),
        "mean_eu": ablation_decompositions["test"][variant].epistemic.mean().item(),
        "mean_logit_variance": ablation_decompositions["test"][variant].logit_variance.mean().item(),
    }
    for variant in ABLATION_VARIANTS
}
print(f"{'posterior':<18}{'accuracy':>10}{'NLL':>10}{'ECE':>10}{'mean EU':>11}{'mean Var[f]':>13}")
for variant, metrics in ablation_quality.items():
    print(
        f"{ABLATION_VARIANTS[variant]:<18}{metrics['accuracy']:>10.4f}{metrics['nll']:>10.4f}"
        f"{metrics['ece']:>10.4f}{metrics['mean_eu']:>11.2e}{metrics['mean_logit_variance']:>13.2e}"
    )


def score_of(set_name: str, variant: str, score: str) -> np.ndarray:
    decomposition = ablation_decompositions[set_name][variant]
    values = decomposition.epistemic if score == "EU" else decomposition.logit_variance
    return values[:, 0].numpy()


def misclassification_auroc(variant: str, score: str, indices: np.ndarray) -> float:
    # Errors are defined by each posterior's own BMA prediction.
    p_bar = ablation_decompositions["test"][variant].p_bar[:, 0].numpy()[indices]
    errors = (p_bar >= UNSAFE_THRESHOLD) != y_test_subset[:, 0].numpy().astype(bool)[indices]
    return auroc(score_of("test", variant, score)[indices], errors)


def shift_auroc(set_name: str, variant: str, score: str, in_indices: np.ndarray, out_indices: np.ndarray) -> float:
    scores_in = score_of("test", variant, score)[in_indices]
    scores_out = score_of(set_name, variant, score)[out_indices]
    is_shifted = np.concatenate([np.zeros(len(scores_in), bool), np.ones(len(scores_out), bool)])
    return auroc(np.concatenate([scores_in, scores_out]), is_shifted)


# Every AUROC for both posteriors, with the paired bootstrap of full-rank minus TFB.
num_test = len(test_subset_indices)
ablation_auroc: dict[str, dict[str, Any]] = {}
for score in ("EU", "Var[f]"):
    ablation_auroc[f"misclassification {score}"] = {
        "tfb": misclassification_auroc("tfb", score, np.arange(num_test)),
        "full_rank": misclassification_auroc("full_rank", score, np.arange(num_test)),
        "full_rank_minus_tfb": bootstrap_difference(
            lambda idx, s=score: misclassification_auroc("full_rank", s, idx[0]),
            lambda idx, s=score: misclassification_auroc("tfb", s, idx[0]),
            [num_test],
            BOOTSTRAP_RESAMPLES,
            SEED,
        ),
    }
    for set_name in ood_datasets:
        num_out = len(ood_datasets[set_name])
        ablation_auroc[f"{set_name} {score}"] = {
            "tfb": shift_auroc(set_name, "tfb", score, np.arange(num_test), np.arange(num_out)),
            "full_rank": shift_auroc(set_name, "full_rank", score, np.arange(num_test), np.arange(num_out)),
            "full_rank_minus_tfb": bootstrap_difference(
                lambda idx, n=set_name, s=score: shift_auroc(n, "full_rank", s, idx[0], idx[1]),
                lambda idx, n=set_name, s=score: shift_auroc(n, "tfb", s, idx[0], idx[1]),
                [num_test, num_out],
                BOOTSTRAP_RESAMPLES,
                SEED,
            ),
        }

print(f"\n{'AUROC':<26}{'TFB':>8}{'full-rank':>11}{'difference [95% CI]':>32}")
for name, row in ablation_auroc.items():
    diff = row["full_rank_minus_tfb"]
    interval = f"{diff['difference']:+.4f} [{diff['ci_low']:+.4f}, {diff['ci_high']:+.4f}]"
    print(f"{name:<26}{row['tfb']:>8.4f}{row['full_rank']:>11.4f}{interval:>32}")

# Epistemic uncertainty in and out of distribution under each posterior, on a log
# scale because the two posteriors may differ by orders of magnitude.
fig, axes = plt.subplots(1, len(ABLATION_VARIANTS), figsize=(14, 4))
for ax, (variant, label) in zip(axes, ABLATION_VARIANTS.items(), strict=True):
    log_eu = {
        set_name: np.log10(by_variant[variant].epistemic[:, 0].numpy() + 1e-12)
        for set_name, by_variant in ablation_decompositions.items()
    }
    pooled = np.concatenate(list(log_eu.values()))
    bins = np.linspace(pooled.min(), max(pooled.max(), pooled.min() + 1e-6), 50)
    for set_name, values in log_eu.items():
        ax.hist(values, bins=bins, alpha=0.5, density=True, label="test subset" if set_name == "test" else set_name)
    ax.set_xlabel(r"$\log_{10}\,\mathrm{EU}$")
    ax.set_title(label)
axes[0].set_ylabel("density")
axes[0].legend()
fig.suptitle("Epistemic uncertainty in and out of distribution: TFB vs full-rank")
plt.tight_layout()
plt.show()

# The ablation writes its own file, so the main summary is never affected by it.
full_rank_summary = {
    "sigma_f_star": sigma_f_star,
    "sigma_q_star": sigma_star,
    "search_trace": full_rank_trace,
    "weight_rms": full_rank_posterior.rms,
    "quality": ablation_quality,
    "auroc": ablation_auroc,
    "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
}
(RESULTS_DIR / "full_rank_ablation.json").write_text(json.dumps(full_rank_summary, indent=2))
print(f"\nWrote {RESULTS_DIR / 'full_rank_ablation.json'}")

## Results

To be written after the Nemotron-Safety-Guard-Dataset-v3 run.